In [ ]:
#准备数据
#DATA 中下载数据dicom_videos

# 划分数据集

In [5]:
#执行数据集分割
import os
os.chdir('/home/mzhao/华为项目/EchoView/stage1_visual_adaptation/ViDEO_DATA')
! python split_data.py

递归查找所有 report.json 文件
查找完毕！
✅ 找到 13 个 report.json 文件
处理报告文件:   0%|                                      | 0/13 [00:00<?, ?it/s]  ✓ /home/mzhao/华为项目/EchoView/stage1_visual_adaptation/ViDEO_DATA/dicom_videos/20240118114734000505695f0c0/report.json
  ✓ /home/mzhao/华为项目/EchoView/stage1_visual_adaptation/ViDEO_DATA/dicom_videos/20240118111000000505695f0c0/report.json
  ✓ /home/mzhao/华为项目/EchoView/stage1_visual_adaptation/ViDEO_DATA/dicom_videos/20240118114825000505695f0c0/report.json
  ✓ /home/mzhao/华为项目/EchoView/stage1_visual_adaptation/ViDEO_DATA/dicom_videos/20240118110659000505695f0c0/report.json
  ✓ /home/mzhao/华为项目/EchoView/stage1_visual_adaptation/ViDEO_DATA/dicom_videos/20240118095503000505695f0c0/report.json
  ✓ /home/mzhao/华为项目/EchoView/stage1_visual_adaptation/ViDEO_DATA/dicom_videos/20240118115009000505695f0c0/report.json
  ✓ /home/mzhao/华为项目/EchoView/stage1_visual_adaptation/ViDEO_DATA/dicom_videos/20240118115019000505695f0c0/report.json
  ✓ /home/mzhao/华为项目/EchoView/stage1_vis

# 构造训练数据json

In [3]:
# 构造GPT消息
import os
os.chdir('/home/mzhao/华为项目/EchoView/stage1_visual_adaptation/sft_builder')
## only_video video_and_mv multi_video all_video video_to_multi_task only_images
! python BuilderMsg.py \
    --diag_item all \
    --bulider_mode only_images 

P: 95 N: 96                                                                     
After balancing: P: 95 N: 95
P: 139 N: 480                                                                   
After balancing: P: 100 N: 100
P: 414 N: 440                                                                   
After balancing: P: 100 N: 100
P: 277 N: 440                                                                   
After balancing: P: 100 N: 100
P: 139 N: 716                                                                   
After balancing: P: 100 N: 100
P: 68 N: 690                                                                    
After balancing: P: 68 N: 68
P: 120 N: 480                                                                   
After balancing: P: 100 N: 100
P: 79 N: 440                                                                    
After balancing: P: 79 N: 79
P: 844 N: 480                                                                   
After balancing: P: 100 N: 10

# 训练视觉模块

In [ ]:
import os
os.chdir('/home/mzhao/华为项目/EchoView/stage1_visual_adaptation/visual_train')
# 指定使用显卡0
os.environ['CUDA_VISIBLE_DEVICES'] = '2'

# 然后运行训练命令
! python train.py \
    --train_json "../SFT_BUILDER/video_to_multi_task/train/visual_multi_task.json" \
    --num_epochs 1 \
    --unfreeze_layers 3

GPU 信息:
CUDA 是否可用: True
当前设备索引: 0
当前设备名称: NVIDIA RTX A6000
可见GPU数量: 1

显卡 0:
  名称: NVIDIA RTX A6000
  显存总量: 47.41 GB
  已分配显存: 0.00 GB
  保留显存: 0.00 GB
Multi-Task Training Configuration:
  Train JSON: ../SFT_BUILDER/video_to_multi_task/train/visual_multi_task.json
  Epochs: 1
  Unfreeze Layers: 3
  CUDA: 0
  Model: /home/mzhao/公共大模型/Qwen2.5-VL-7B-Instruct
Training Multi-Task Model
Epochs: 1, Unfreeze Layers: 3

Loading Qwen2.5-VL model...
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████████████| 5/5 [00:03<00:00,  1.37it/s]
✓ Unfreezing last 3 layers (blocks 29-31)
📊 Vision encoder: 59,106,600/676,550,144 (8.74%)
📊 Classifier head: 100,380 params
📊 Regression head: 25,095 params
✅ Total trainable: 59,232,075
🎯 Head params占比: 0.21%
Model created with hidden size: 3584
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking c

# 测试视觉模块

In [ ]:
# 测试视觉模块的效果
import os
os.chdir('/home/mzhao/华为项目/EchoView/stage1_visual_adaptation/visual_train')

# 指定使用显卡0
os.environ['CUDA_VISIBLE_DEVICES'] = '5'
! python predict.py \
    --test_json "../SFT_BUILDER/video_to_multi_task/test/visual_multi_task.json" \
    --checkpoint "../weights/pth/best_multitask_model.pth" \
    --batch_size 2 \
    --num_frames 8
    

GPU 信息:
CUDA 是否可用: True
当前设备索引: 0
当前设备名称: NVIDIA RTX 5880 Ada Generation
可见GPU数量: 1

显卡 0:
  名称: NVIDIA RTX 5880 Ada Generation
  显存总量: 47.38 GB
  已分配显存: 0.00 GB
  保留显存: 0.00 GB
MULTI-TASK EVALUATION CONFIGURATION
  Test JSON:         ../SFT_BUILDER/video_to_multi_task/test/visual_multi_task.json
  Checkpoint:        pth/best_multitask_model.pth
  Model:             /home/mzhao/公共大模型/Qwen2.5-VL-7B-Instruct
  Batch Size:        2
  Num Frames:        8
  Binary Tasks:      28
  Regression Tasks:  7

Using device: cuda

Loading processor...
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.

Loading test data from ../SFT_BUILDER/video_to_multi_task/test/v

# 合并视觉模块权重

In [ ]:
#如果确定上面训练的视觉模块没问题，就在Qwen2.5VL模型中加载然后保存成Huggface模式
import os
os.chdir('/home/mzhao/华为项目/EchoView/stage1_visual_adaptation/visual_train')
os.environ['CUDA_VISIBLE_DEVICES'] = '2'
! python merged_pth.py \
    --checkpoint_path "../weights/pth/best_multitask_model.pth" \
    --base_model_path "/home/mzhao/公共大模型/Qwen2.5-VL-7B-Instruct" \
    --save_dir "../weights/qwen2.5-vl-7b-visual_finetuned"

GPU 信息:
CUDA 是否可用: True
当前设备索引: 0
当前设备名称: NVIDIA RTX A6000
可见GPU数量: 1

显卡 0:
  名称: NVIDIA RTX A6000
  显存总量: 47.41 GB
  已分配显存: 0.00 GB
  保留显存: 0.00 GB
🚀 Qwen2.5-VL Model Merger
Checkpoint: pth/best_multitask_model.pth
Base model: /home/mzhao/公共大模型/Qwen2.5-VL-7B-Instruct
Save dir: pth/qwen2.5-vl-7b-visual_finetuned
Device: cuda
Torch dtype: auto
Safe serialization: True
📂 Loading checkpoint from: pth/best_multitask_model.pth
Checkpoint keys: dict_keys(['epoch', 'model_state_dict', 'optimizer_state_dict', 'val_binary_acc', 'val_reg_mae', 'val_reg_rmse'])
Epoch: 0
Val Acc: N/A

📊 Extracted 390 visual weight keys
键名转换示例:
  visual.patch_embed.proj.weight → patch_embed.proj.weight
  visual.blocks.0.norm1.weight → blocks.0.norm1.weight
  visual.blocks.0.norm2.weight → blocks.0.norm2.weight

📦 Loading base model from: /home/mzhao/公共大模型/Qwen2.5-VL-7B-Instruct
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████████████| 5/5 [00:03<00:00,  1.43it/s]

🔄 Merg

# 开始训练语言模块，注意修改权重和训练数据路径

In [24]:
# 训练Qwen2.5VL
import os
os.chdir('/home/mzhao/华为项目/EchoView/stage2_experts/vlm_train_alt')
os.environ['CUDA_VISIBLE_DEVICES'] = '2'

! bash vlm_train.sh  #注意修改参数，

训练配置
 train_mode      : only_video
 llm             : ../VISUAL_TRAIN/pth/qwen2.5-vl-7b-visual_finetuned
 data_json       : ../SFT_BUILDER/only_video/train/merged_train_list.json
 output_dir      : vlm_weight/only_video
 lr              : 1e-4
 batch_size      : 8
 grad_accum      : 2
 effective_batch : 16

GPU 信息:
CUDA 是否可用: True
当前设备索引: 0
当前设备名称: NVIDIA RTX A6000
可见GPU数量: 1

显卡 0:
  名称: NVIDIA RTX A6000
  显存总量: 47.41 GB
  已分配显存: 0.00 GB
  保留显存: 0.00 GB
4.57.3
加载模型权重: ../VISUAL_TRAIN/pth/qwen2.5-vl-7b-visual_finetuned
Loading checkpoint shards: 100%|██████████████████| 4/4 [00:03<00:00,  1.13it/s]

=== 调试：检查模型加载 ===
Model type: <class 'transformers.models.qwen2_5_vl.modeling_qwen2_5_vl.Qwen2_5_VLForConditionalGeneration'>
Has visual: True
  patch_embed.proj.weight: torch.Size([0])
Visual parameter count: 0
启动2.5VL模型
the initlized model is ../VISUAL_TRAIN/pth/qwen2.5-vl-7b-visual_finetuned the class is Qwen2_5_VLForConditionalGeneration
The image processor of type `Qwen2VLImageProcesso

# 合并权重

In [4]:
# 把训练好的lora 合并到原模型，方便调用
import os
os.chdir('/home/mzhao/华为项目/EchoView/stage2_experts/vlm_train_alt')
os.environ['CUDA_VISIBLE_DEVICES'] = '2'
! python merged_lora.py \
    --base_model_path "../weights/vlm_best_merged_model_only_video" \
    --lora_checkpoint "../weights/vlm_lora/video_and_mv/checkpoint-3000" \
    --save_dir "../weights/vlm_best_merged_model_video_and_mv"

GPU 信息:
CUDA 是否可用: True
当前设备索引: 0
当前设备名称: NVIDIA RTX A6000
可见GPU数量: 1

显卡 0:
  名称: NVIDIA RTX A6000
  显存总量: 47.41 GB
  已分配显存: 0.00 GB
  保留显存: 0.00 GB
加载 LoRA 模型
基座模型: ../weights/vlm_best_merged_model_only_video
LoRA路径: ../weights/vlm_lora/video_and_mv/checkpoint-3000
保存路径: ../weights/vlm_best_merged_model_video_and_mv

📦 加载基座模型...
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████████████| 4/4 [00:02<00:00,  1.39it/s]
🔧 加载 LoRA 适配器...
🔄 融合 LoRA 权重...
💾 保存到: ../weights/vlm_best_merged_model_video_and_mv
The tokenizer you are loading from '../weights/vlm_best_merged_model_only_video' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
✅ 完成！

使用方式:
model = Qwen2_5_VLForConditionalGeneration.from_pretrained('../weights/vlm_

# 测试模型

In [6]:
# 测试推理
import os
os.chdir('/home/mzhao/华为项目/EchoView/stage2_experts/vlm_train_alt')
os.environ['CUDA_VISIBLE_DEVICES'] = '2'
! python inference.py \
    --model_path "../weights/vlm_best_merged_model_video_and_mv" \
    --processor_path "/home/mzhao/公共大模型/Qwen2.5-VL-7B-Instruct" \
    --diag_item_config "../SFT_BUILDER/diag_info.json" \
    --lora_path "" \
    --data_path "../SFT_BUILDER/video_and_mv/test/" \
    --diag_item all \
    --data_root "" \
    --sample_size 200 \
    --num_frames 16 \
    --max_new_tokens 2048 \
    --print_result False \
    --save_results "../inference_result/video_and_mv"

GPU 信息:
CUDA 是否可用: True
当前设备索引: 0
当前设备名称: NVIDIA RTX A6000
可见GPU数量: 1

显卡 0:
  名称: NVIDIA RTX A6000
  显存总量: 47.41 GB
  已分配显存: 0.00 GB
  保留显存: 0.00 GB
CUDA_VISIBLE_DEVICES = 2
torch.cuda.is_available(): True
GPU设备: NVIDIA RTX A6000
显存总量: 47.41 GB
当前空闲显存: 0.00 GB
正在加载模型...
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████████████| 4/4 [00:02<00:00,  1.39it/s]
lora设置为不加载！
模型加载完成!
Loading info: {'missing_keys': [], 'unexpected_keys': [], 'mismatched_keys': [], 'error_msgs': []}
模型设备: cuda:0
正在加载数据: ../SFT_BUILDER/video_and_mv/test/心包积液.json
总样本数: 190
采样样本数: 190
Pro

# 统计模型推理结果

In [28]:
#统计指标
import os
os.chdir('/home/mzhao/华为项目/EchoView/stage1_visual_adaptation/inference_result')
! python show_metrics.py \
    -f "only_video/" \
    -o "only_video/"


📁 处理文件夹: only_video/
  二尖瓣反流                               样本:   10  准确率:0.400  F1:0.500
  瓣环钙化                                样本:   10  准确率:0.500  F1:0.286
  心尖部增厚                               样本:   10  准确率:0.700  F1:0.727
  左房增大                                样本:   10  准确率:0.400  F1:0.571
  主动脉瓣反流                              样本:   10  准确率:0.500  F1:0.545
  二叶式主动脉瓣                             样本:   10  准确率:0.700  F1:0.769
  左室增大                                样本:   10  准确率:0.500  F1:0.667
  主动脉瓣狭窄                              样本:   10  准确率:0.500  F1:0.545
  二尖瓣脱垂                               样本:   10  准确率:0.300  F1:0.462
  三尖瓣反流                               样本:   10  准确率:0.400  F1:0.500
  右房增大                                样本:   10  准确率:0.500  F1:0.615
  主动脉瓣钙化                              样本:   10  准确率:0.400  F1:0.250
  肥厚型心肌病                              样本:   10  准确率:0.500  F1:0.545
  收缩活动减弱                              样本:   10  准确率:0.300  F1:0.222
  起搏器                      

# 调整阈值并统计结果

In [11]:
import os
os.chdir('/home/mzhao/华为项目/EchoView/stage1_visual_adaptation/inference_result')
! python cmp.py \
    --json_floder "video_and_mv" \
    --output "video_and_mv/my_results.xlsx"


📁 处理文件: inference_二尖瓣反流.json
📊 数据概况
总样本数: 200
阳性样本 (是): 88
阴性样本 (否): 112
prob_y 范围: [0.0000016196, 1.0000000000]
prob_y 均值: 0.4370598276

📌 标签分布:
   '是' (阳性): 88 个
   '否' (阴性): 112 个

🔍 寻找最优阈值 (优化指标: F1)
🏆 最优阈值 (F1最大化): 0.8066789612
   对应的F1分数: 0.988506

📋 使用最优阈值 (T=0.806679) 评估
混淆矩阵:
   ┌─────────────┐
   │  TP: 86   FP: 0  │
   │  FN: 2    TN: 112 │
   └─────────────┘

📊 性能指标:
   ├─ 准确率 (Accuracy):  0.990000  (99.00%)
   ├─ 精确率 (Precision): 1.000000  (100.00%)
   ├─ 召回率 (Recall):    0.977273  (97.73%)
   ├─ F1分数:             0.988506  (98.85%)
   └─ AUC:                0.995079  (99.51%)


📁 处理文件: inference_瓣环钙化.json
📊 数据概况
总样本数: 200
阳性样本 (是): 96
阴性样本 (否): 104
prob_y 范围: [0.0000357797, 1.0000000000]
prob_y 均值: 0.4737452040

📌 标签分布:
   '是' (阳性): 96 个
   '否' (阴性): 104 个

🔍 寻找最优阈值 (优化指标: F1)
🏆 最优阈值 (F1最大化): 0.3831516285
   对应的F1分数: 0.958333

📋 使用最优阈值 (T=0.383152) 评估
混淆矩阵:
   ┌─────────────┐
   │  TP: 92   FP: 4  │
   │  FN: 4    TN: 100 │
   └─────────────┘

📊 性能指标:
   ├─ 准确率 (Accuracy